In [48]:
import re
import numpy as np
import pandas as pd

from gensim.models import Word2Vec, Doc2Vec
from gensim.models.doc2vec import TaggedDocument

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.cluster import KMeans


In [50]:

# =========================
# 1) Load Excel file
# =========================

EXCEL_PATH = "articles_all_categories_one_file.xlsx"

def load_excel_all_sheets(path: str) -> pd.DataFrame:
    all_sheets = pd.read_excel(path, sheet_name=None)

    frames = []
    for sheet_name, df in all_sheets.items():
        df = df.copy()

        # If Category column does not exist, use sheet name as category
        if "Category" not in df.columns:
            df["Category"] = sheet_name

        # Keep only useful columns
        df = df[["Category", "Article"]]
        frames.append(df)

    data = pd.concat(frames, ignore_index=True)
    data = data.dropna(subset=["Article", "Category"])
    data["Article"] = data["Article"].astype(str)
    data["Category"] = data["Category"].astype(str)

    return data


df = load_excel_all_sheets(EXCEL_PATH)

print("Dataset shape:", df.shape)
print(df["Category"].value_counts())

df

Dataset shape: (500, 2)
Category
صحة       103
رياضة     100
اقتصاد    100
غذاء      100
سياسة      97
Name: count, dtype: int64


,Category,Article
0,صحة,ناقش البرلمان اليوم مشروع قانون جديد يتعلق بال...
1,صحة,ناقش البرلمان اليوم مشروع قانون جديد يتعلق بال...
2,صحة,أكد الوزير أن الإصلاح الإداري يمثل أولوية في ب...
3,سياسة,شهدت العاصمة اجتماعاً سياسياً لبحث القضايا الإ...
4,سياسة,أعلنت الحكومة عن خطة لتعزيز العلاقات الدبلوماس...
...,...,...
495,غذاء,تناولت المقالة فوائد الفواكه الموسمية وأثرها ع...
496,غذاء,تشهد المطاعم المحلية إقبالاً متزايداً على الأط...
497,غذاء,يفضل الكثير من الناس استخدام زيت الزيتون في إع...
498,غذاء,تشهد المطاعم المحلية إقبالاً متزايداً على الأط...


In [51]:

# =========================
# 2) Arabic text cleaning + tokenization
# =========================

arabic_stopwords = {
    "في", "من", "على", "إلى", "عن", "أن", "إن", "كان", "كانت", "هذا", "هذه",
    "ذلك", "تلك", "هو", "هي", "هم", "كما", "قد", "مع", "بين", "بعد", "قبل",
    "أو", "و", "ثم", "لا", "ما", "لم", "لن", "كل", "أي", "حتى", "لدى", "عند",
    "التي", "الذي", "الذين", "ضمن", "حيث", "الى"
}

def clean_arabic_text(text: str) -> str:
    text = str(text)

    # Remove Arabic diacritics
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)

    # Normalize some Arabic letters
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)

    # Keep Arabic letters and spaces only
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text: str) -> list[str]:
    text = clean_arabic_text(text)
    tokens = text.split()

    # Remove short words and stopwords
    tokens = [
        token for token in tokens
        if len(token) > 2 and token not in arabic_stopwords
    ]

    return tokens

df["tokens"] = df["Article"].apply(tokenize)
df

,Category,Article,tokens
0,صحة,ناقش البرلمان اليوم مشروع قانون جديد يتعلق بال...,"[ناقش, البرلمان, اليوم, مشروع, قانون, جديد, يت..."
1,صحة,ناقش البرلمان اليوم مشروع قانون جديد يتعلق بال...,"[ناقش, البرلمان, اليوم, مشروع, قانون, جديد, يت..."
2,صحة,أكد الوزير أن الإصلاح الإداري يمثل أولوية في ب...,"[اكد, الوزير, الاصلاح, الاداري, يمثل, اولويه, ..."
3,سياسة,شهدت العاصمة اجتماعاً سياسياً لبحث القضايا الإ...,"[شهدت, العاصمه, اجتماعا, سياسيا, لبحث, القضايا..."
4,سياسة,أعلنت الحكومة عن خطة لتعزيز العلاقات الدبلوماس...,"[اعلنت, الحكومه, خطه, لتعزيز, العلاقات, الدبلو..."
...,...,...,...
495,غذاء,تناولت المقالة فوائد الفواكه الموسمية وأثرها ع...,"[تناولت, المقاله, فوايد, الفواكه, الموسميه, وا..."
496,غذاء,تشهد المطاعم المحلية إقبالاً متزايداً على الأط...,"[تشهد, المطاعم, المحليه, اقبالا, متزايدا, علي,..."
497,غذاء,يفضل الكثير من الناس استخدام زيت الزيتون في إع...,"[يفضل, الكثير, الناس, استخدام, زيت, الزيتون, ا..."
498,غذاء,تشهد المطاعم المحلية إقبالاً متزايداً على الأط...,"[تشهد, المطاعم, المحليه, اقبالا, متزايدا, علي,..."


In [52]:

# Remove empty token rows if any
df = df[df["tokens"].map(len) > 0].reset_index(drop=True)

print("\nAfter tokenization:", df.shape)




After tokenization: (500, 3)


In [54]:

# =========================
# 3) Encode labels
# =========================

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["Category"])

X_tokens = df["tokens"].tolist()
y = df["label"].values

X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
    X_tokens,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



In [55]:
X_train_tokens[0]

['تناولت',
 'الندوه',
 'الطبيه',
 'احدث',
 'طرق',
 'علاج',
 'الامراض',
 'المزمنه',
 'ينصح',
 'الاطباء',
 'بممارسه',
 'الرياضه',
 'بشكل',
 'منتظم',
 'للحفاظ',
 'علي',
 'صحه',
 'القلب',
 'اطلقت',
 'وزاره',
 'الصحه',
 'حمله',
 'توعيه',
 'حول',
 'اهميه',
 'اللقاحات',
 'للاطفال',
 'اكدت',
 'الدراسات',
 'النوم',
 'الجيد',
 'يساعد',
 'علي',
 'تحسين',
 'التركيز',
 'والصحه',
 'النفسيه',
 'يشدد',
 'المختصون',
 'علي',
 'اهميه',
 'شرب',
 'الماء',
 'للحفاظ',
 'علي',
 'نشاط',
 'الجسم']

In [56]:

# =========================
# 4) Word2Vec model
# =========================

word2vec_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,          # 1 = Skip-Gram, 0 = CBOW
    epochs=30,
    seed=42
)

word2vec_model.save("word2vec_articles.model")


def document_vector_word2vec(tokens: list[str], model: Word2Vec) -> np.ndarray:
    vectors = []

    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


X_train_w2v = np.array([document_vector_word2vec(tokens, word2vec_model) for tokens in X_train_tokens])
X_test_w2v = np.array([document_vector_word2vec(tokens, word2vec_model) for tokens in X_test_tokens])



In [59]:
X_train_w2v.shape

(400, 100)

In [61]:
X_train_w2v[0]

array([-0.16183434,  0.3140848 ,  0.12781364,  0.19407125, -0.25320628,
        0.09664835,  0.14637586,  0.28362638, -0.34547526, -0.14304979,
       -0.2675366 ,  0.30331862, -0.12383527,  0.6398466 , -0.27119666,
       -0.40952685,  0.14089602, -0.30212644,  0.06121007,  0.04437622,
        0.38186297,  0.69214535,  0.07299211,  0.10480558, -0.20631357,
       -0.35601178, -0.5548698 ,  0.4470753 , -0.44577035, -0.41754073,
       -0.17163445, -0.02091722,  0.25009102,  0.16297889,  0.40624985,
        0.2174792 ,  0.19062099,  0.1812023 , -0.14907788,  0.09276768,
       -0.1400679 , -0.24735843,  0.42655858, -0.09116187, -0.09122305,
        0.05466088,  0.05158304, -0.0082388 ,  0.11526121, -0.17273243,
        0.08475907, -0.537821  , -0.15077071,  0.33312744, -0.4679049 ,
       -0.25418678, -0.31160837,  0.13632236,  0.30402023,  0.45609486,
       -0.10446225,  0.20531338, -0.14622506, -0.25712037,  0.5893771 ,
       -0.06568535,  0.04851257, -0.08598211, -0.3384227 , -0.04

In [63]:

# =========================
# 5) Classification using Word2Vec vectors
# =========================

w2v_classifier = LogisticRegression(max_iter=1000, random_state=42)
w2v_classifier.fit(X_train_w2v, y_train)

y_pred_w2v = w2v_classifier.predict(X_test_w2v)

print("\n==============================")
print("Word2Vec Classification Result")
print("==============================")
print("Accuracy:", accuracy_score(y_test, y_pred_w2v))
print(classification_report(
    y_test,
    y_pred_w2v,
    target_names=label_encoder.classes_
))




Word2Vec Classification Result
Accuracy: 1.0
              precision    recall  f1-score   support

      اقتصاد       1.00      1.00      1.00        20
       رياضة       1.00      1.00      1.00        20
       سياسة       1.00      1.00      1.00        19
         صحة       1.00      1.00      1.00        21
        غذاء       1.00      1.00      1.00        20

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [64]:
# =========================
# 8) Predict new article category
# =========================

def predict_with_word2vec(text: str) -> str:
    tokens = tokenize(text)
    vector = document_vector_word2vec(tokens, word2vec_model).reshape(1, -1)
    predicted_label = w2v_classifier.predict(vector)[0]
    return label_encoder.inverse_transform([predicted_label])[0]




In [65]:

new_text = """
تشهد سوريا نموا استثماريا في جميع انحاء البلد
"""

print("\n==============================")
print("Prediction for New Text")
print("==============================")
print("Word2Vec Prediction:", predict_with_word2vec(new_text))



Prediction for New Text
Word2Vec Prediction: اقتصاد


In [66]:
# 9) Clustering using document vectors
# =========================

# We will cluster all documents using Word2Vec average vectors
all_vectors_w2v = np.array([
    document_vector_word2vec(tokens, word2vec_model)
    for tokens in df["tokens"]
])
all_vectors_w2v.shape

(500, 100)

In [67]:

num_clusters = 5

kmeans = KMeans(
    n_clusters=num_clusters,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(all_vectors_w2v)
df.head()

,Category,Article,tokens,label,cluster
0,صحة,ناقش البرلمان اليوم مشروع قانون جديد يتعلق بال...,"[ناقش, البرلمان, اليوم, مشروع, قانون, جديد, يت...",3,2
1,صحة,ناقش البرلمان اليوم مشروع قانون جديد يتعلق بال...,"[ناقش, البرلمان, اليوم, مشروع, قانون, جديد, يت...",3,2
2,صحة,أكد الوزير أن الإصلاح الإداري يمثل أولوية في ب...,"[اكد, الوزير, الاصلاح, الاداري, يمثل, اولويه, ...",3,2
3,سياسة,شهدت العاصمة اجتماعاً سياسياً لبحث القضايا الإ...,"[شهدت, العاصمه, اجتماعا, سياسيا, لبحث, القضايا...",2,2
4,سياسة,أعلنت الحكومة عن خطة لتعزيز العلاقات الدبلوماس...,"[اعلنت, الحكومه, خطه, لتعزيز, العلاقات, الدبلو...",2,2


In [68]:

# =========================
# 10) Statistics: category distribution inside each cluster
# =========================

cluster_counts = pd.crosstab(
    df["cluster"],
    df["Category"]
)

cluster_percentages = pd.crosstab(
    df["cluster"],
    df["Category"],
    normalize="index"
) * 100

print("\n==============================")
print("Cluster Counts")
print("==============================")
print(cluster_counts)

print("\n==============================")
print("Cluster Percentages")
print("==============================")
print(cluster_percentages.round(2))





Cluster Counts
Category  اقتصاد  رياضة  سياسة  صحة  غذاء
cluster                                  
0              0    100      0    0     0
1              0      0      0  100     0
2              0      0     97    3     0
3              0      0      0    0   100
4            100      0      0    0     0

Cluster Percentages
Category  اقتصاد  رياضة  سياسة    صحة   غذاء
cluster                                     
0            0.0  100.0    0.0    0.0    0.0
1            0.0    0.0    0.0  100.0    0.0
2            0.0    0.0   97.0    3.0    0.0
3            0.0    0.0    0.0    0.0  100.0
4          100.0    0.0    0.0    0.0    0.0


In [69]:
# Save results to Excel
with pd.ExcelWriter("classification_clustering_results.xlsx", engine="openpyxl") as writer:
    df[["Category", "Article", "cluster"]].to_excel(
        writer,
        sheet_name="Documents_With_Clusters",
        index=False
    )

    cluster_counts.to_excel(
        writer,
        sheet_name="Cluster_Counts"
    )

    cluster_percentages.round(2).to_excel(
        writer,
        sheet_name="Cluster_Percentages"
    )

print("\nResults saved to: classification_clustering_results.xlsx")



Results saved to: classification_clustering_results.xlsx
